In [14]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from sklearn.random_projection import GaussianRandomProjection
import re
from collections import defaultdict

class ASHD:
    def __init__(self, k=7, dim=256, iters=25, a=0.5, lam=0.01, gam=0.1):
        self.k = k
        self.dim = dim
        self.iters = iters
        self.a = a
        self.lam = lam
        self.gam = gam
        self.eps = 1e-10
        self.w_hist = []

    def load_data(self, p_file, c_file, t_file):
        p_list = []
        with open(p_file, 'r') as f:
            for line in f:
                if ';paper;' in line:
                    pid = line.split(';')[0]
                    p_list.append(pid)
        
        self.p_list = sorted(list(set(p_list)))
        self.p2i = {pid: i for i, pid in enumerate(self.p_list)}
        n = len(self.p_list)
        
        print(f"  Papers: {n}")
        
        p_topics = {}
        with open(t_file, 'r') as f:
            content = f.read()
            matches = re.findall(r'(\w+)\((\d+)\)=1', content)
            t_ids = {}
            curr = 0
            for t_name, pid in matches:
                if t_name not in t_ids:
                    t_ids[t_name] = curr
                    curr += 1
                if pid in self.p2i:
                    p_topics[pid] = t_ids[t_name]
        
        print(f"  Topics: {len(t_ids)}")
        
        pos = self._build_pos(c_file)
        neg = self._build_neg_v3(c_file, t_file)
        
        print(f"  Positive hyperedges: {len(pos)}")
        print(f"  Negative hyperedges: {len(neg)}")
        
        all_e = pos + neg
        signs = np.array([1.0] * len(pos) + [-1.0] * len(neg))
        
        rows, cols = [], []
        for j, nodes in enumerate(all_e):
            for i in nodes:
                rows.append(i)
                cols.append(j)
        
        h = sparse.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(n, len(all_e)))
        
        y = np.full(n, -1)
        for i, pid in enumerate(self.p_list):
            if pid in p_topics:
                y[i] = p_topics[pid]
        
        x_init = np.random.normal(0, 0.01, (n, self.dim))
        x_init = normalize(x_init, norm='l2', axis=1)
        
        return h, signs, y, x_init

    def _build_pos(self, c_file):
        pos_e = defaultdict(set)
        with open(c_file, 'r') as f:
            content = f.read()
            cites = re.findall(r'Cite\((\d+),(\d+)\)=1', content)
            for s, t in cites:
                if s in self.p2i and t in self.p2i:
                    pos_e[s].add(self.p2i[s])
                    pos_e[s].add(self.p2i[t])
        
        return [list(nodes) for nodes in pos_e.values() if len(nodes) > 1]

    def _build_neg_v3(self, c_file, t_file):
        cite_g = defaultdict(set)
        with open(c_file, 'r') as f:
            content = f.read()
            cites = re.findall(r'Cite\((\d+),(\d+)\)=1', content)
            for s, t in cites:
                if s in self.p2i and t in self.p2i:
                    cite_g[self.p2i[s]].add(self.p2i[t])
                    cite_g[self.p2i[t]].add(self.p2i[s])
        
        p_topics = {}
        with open(t_file, 'r') as f:
            content = f.read()
            matches = re.findall(r'(\w+)\((\d+)\)=1', content)
            for t_name, pid in matches:
                if pid in self.p2i:
                    p_topics[self.p2i[pid]] = t_name
        
        topic_ps = defaultdict(list)
        for i, t in p_topics.items():
            topic_ps[t].append(i)
        
        neg_e = []
        for t, ps in topic_ps.items():
            if len(ps) < 2:
                continue
            for i in range(len(ps)):
                for j in range(i+1, min(i+6, len(ps))):
                    p1, p2 = ps[i], ps[j]
                    if p2 not in cite_g.get(p1, set()) and p1 not in cite_g.get(p2, set()):
                        neg_e.append([p1, p2])
        
        return neg_e

    def fit_predict(self, h, signs, x_init, seed=42):
        n, m = h.shape
        np.random.seed(seed)
        
        x = x_init.copy()
        w = signs.astype(float).copy()
        
        de = np.array(h.sum(axis=0)).flatten() + self.eps
        de_inv = sparse.diags(1.0 / de)

        print(f"  Embedding dim: {self.dim}")
        print(f"  Iterations: {self.iters}")
        print("=" * 60)
        
        self.w_hist = []
        
        for t in range(self.iters):
            w_abs = np.abs(w)
            dv = np.array(h.dot(w_abs)).flatten() + self.eps
            dv_inv_sqrt = sparse.diags(1.0 / np.sqrt(dv))
            
            wd = sparse.diags(w)
            tmp = dv_inv_sqrt.dot(x)
            tmp = h.T.dot(tmp)
            tmp = de_inv.dot(tmp)
            tmp = wd.dot(tmp)
            tmp = h.dot(tmp)
            x_new = dv_inv_sqrt.dot(tmp)
            
            x = x_new - self.lam * x_new
            x = normalize(x, norm='l2', axis=1)
            
            self.w_hist.append(w.copy())
            
            if t > 0 and t % 2 == 0:
                for j in range(m):
                    idx = h.getcol(j).indices
                    if len(idx) > 1:
                        sims = cosine_similarity(x[idx])
                        avg = (np.sum(sims) - len(idx)) / (len(idx) * (len(idx) - 1) + self.eps)
                        
                        if w[j] > 0:
                            w[j] = np.clip(w[j] + self.a * avg, 0.1, 1.0)
                        else:
                            w[j] = np.clip(w[j] - self.a * avg, -1.0, -0.1)
            
            if (t + 1) % 5 == 0:
                print(f"  Iteration {t+1}/{self.iters}")
        
        km = KMeans(n_clusters=self.k, n_init=20, random_state=seed)
        lbls = km.fit_predict(x)
        
        return lbls, x

if __name__ == "__main__":
    model = ASHD(k=7, dim=256, iters=25, a=0.5, lam=0.01, gam=0.1)
    
    h, signs, y, x_init = model.load_data('papers_dataset.txt', 'citations.txt', 'topics.txt')
    
    lbls, x = model.fit_predict(h, signs, x_init)
    
    mask = y != -1
    ari = adjusted_rand_score(y[mask], lbls[mask])
    nmi = normalized_mutual_info_score(y[mask], lbls[mask])
    

    print(f"ARI: {ari:.4f}")
    print(f"NMI: {nmi:.4f}")

  Papers: 11881
  Topics: 79
  Positive hyperedges: 9548
  Negative hyperedges: 55889
  Embedding dim: 256
  Iterations: 25
  Iteration 5/25
